In [11]:
from dataclasses import dataclass

@dataclass
class WingParams:
    R: float        = 0.0                  # wing length [m]
    c: float        = 0.0             # mean chord [m]
    # Non-dimensional radius of moments
    r33: float      = 0.0
    r22: float      = 0.0
    r11: float      = 0.0
    r00: float      = 0.0
    # 压力中心相关的几何积分
    z_cp2: float    = 0.0
    z_cp1: float    = 0.0
    z_cp0: float    = 0.0
    # rotation damping coefficient
    z_rd: float     = 0.0
    # constant parameters
    rho: float      = 0.0      # air density [kg/m³]
    d0: float       = 0.0               # half shoulder width [m]
    ds: float       = 0.0             # stroke plane offset [m]

In [12]:
import math
import json
from dataclasses import asdict
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [13]:
polygon = np.array([
    [0.0,  0.0],   # upper-left / wing root / hinge
    [70.0, 0.0],   # leading-edge tip
    [70.0, 20.0],  # trailing-edge tip
    [0.0,  20.0],  # trailing-edge root
])

length_unit = 'mm'             # 'm', 'cm', or 'mm'
hinge_origin = np.array([0.0, 0.0])
span_direction = np.array([1.0, 0.0])
chord_direction = np.array([0.0, 1.0])

wing_mass = 0.0626             # grams; set None to omit mass properties
air_density = 1.225            # kg/m^3
shoulder_half_width = 13.95     # in length_unit; aero.py d0
stroke_plane_offset = 18.9913   # in length_unit; aero.py ds

quadrature_order = 24           # Gauss points per vertex-to-vertex interval
axis_tolerance = 0.01           # fraction of mean chord for compatibility warning

In [14]:
UNIT_TO_M = {'m': 1.0, 'cm': 1e-2, 'mm': 1e-3}

def _unit(v, name):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    if n == 0:
        raise ValueError(f'{name} must be non-zero')
    return v / n

def _validate_simple_polygon(poly):
    poly = np.asarray(poly, dtype=float)
    if poly.ndim != 2 or poly.shape[1] != 2 or len(poly) < 3:
        raise ValueError('polygon must have shape (N, 2), N >= 3')
    if not np.isfinite(poly).all():
        raise ValueError('polygon contains NaN or infinity')
    if np.allclose(poly[0], poly[-1]):
        poly = poly[:-1]
    area2 = np.sum(poly[:, 0] * np.roll(poly[:, 1], -1) -
                   np.roll(poly[:, 0], -1) * poly[:, 1])
    if abs(area2) < 1e-14:
        raise ValueError('polygon area is zero')
    return poly

def _vertical_intervals(poly_local, r, eps=1e-12):
    # Even-odd fill rule. Half-open edge handling avoids double-counting vertices.
    ys = []
    for a, b in zip(poly_local, np.roll(poly_local, -1, axis=0)):
        r0, y0 = a; r1, y1 = b
        if abs(r1 - r0) <= eps:
            continue
        if min(r0, r1) <= r < max(r0, r1):
            t = (r - r0) / (r1 - r0)
            ys.append(y0 + t * (y1 - y0))
    ys.sort()
    if len(ys) % 2:
        raise ValueError('invalid/self-intersecting polygon near span coordinate ' + str(r))
    return list(zip(ys[0::2], ys[1::2]))

def calculate_wing_params(polygon, hinge_origin, span_direction, chord_direction,
                          *, length_unit='mm', wing_mass_g=None, rho=1.225,
                          d0=0.0, ds=0.0, quadrature_order=24, axis_tolerance=0.01):
    poly = _validate_simple_polygon(polygon)
    if length_unit not in UNIT_TO_M:
        raise ValueError(f'length_unit must be one of {tuple(UNIT_TO_M)}')
    e_r = _unit(span_direction, 'span_direction')
    e_y = _unit(chord_direction, 'chord_direction')
    if abs(np.dot(e_r, e_y)) > 1e-8:
        raise ValueError('span_direction and chord_direction must be perpendicular')
    local = np.column_stack(((poly - hinge_origin) @ e_r, (poly - hinge_origin) @ e_y))
    scale = UNIT_TO_M[length_unit]
    local *= scale
    tol = 1e-10 * max(1.0, np.ptp(local[:, 0]))
    if local[:, 0].min() < -tol:
        raise ValueError('part of the wing lies behind the hinge origin along the span axis')
    R = float(local[:, 0].max())
    if R <= 0:
        raise ValueError('wing span R must be positive')

    # Split at every polygon vertex; edges and slice endpoints are linear inside each interval.
    breaks = np.unique(np.clip(local[:, 0], 0.0, R))
    if breaks[0] > 0: breaks = np.r_[0.0, breaks]
    if breaks[-1] < R: breaks = np.r_[breaks, R]
    gx, gw = np.polynomial.legendre.leggauss(quadrature_order)
    rows = []
    for lo, hi in zip(breaks[:-1], breaks[1:]):
        if hi - lo <= tol: continue
        for x, w in zip(0.5*(hi-lo)*gx + 0.5*(hi+lo), 0.5*(hi-lo)*gw):
            intervals = _vertical_intervals(local, float(x))
            chord = sum(y1-y0 for y0, y1 in intervals)
            rows.append((float(x), float(w), intervals, chord))
    if not rows: raise ValueError('no wing area found in r >= 0')

    integ = lambda f: sum(w*f(r, ints, c) for r, w, ints, c in rows)
    area = integ(lambda r, ints, c: c)
    cbar = area / R
    radial = [integ(lambda r, ints, c, k=k: c*r**k)/(cbar*R**(k+1)) for k in range(4)]
    # aero.py factors d_cp(alpha) outside the integral: z_k = integral(c_hat^2 r_hat^k).
    zcp = [integ(lambda r, ints, c, k=k: c*c*r**k)/(cbar**2*R**(k+1)) for k in range(3)]
    # Whitney & Wood eqs. (2.37)-(2.39); aero.py uses z_rd = 4*Y_rd.
    def damping_slice(r, ints, c):
        return sum(abs(y1)*y1**3 - abs(y0)*y0**3 for y0, y1 in ints)
    z_rd = integ(damping_slice)/(cbar**4 * R)

    # Full paper moment is A_k - d_cp*B_k. A_k must vanish for exact aero.py compatibility.
    offset_terms = [integ(lambda r, ints, c, k=k:
                    sum(y0*(y1-y0) for y0, y1 in ints)*r**k)
                    /(cbar**2*R**(k+1)) for k in range(3)]
    max_axis_offset = max(abs(v) for v in offset_terms)
    compatible = max_axis_offset <= axis_tolerance * max(max(zcp), 1e-15)

    params = WingParams(R=R, c=cbar, r33=radial[3], r22=radial[2],
                        r11=radial[1], r00=radial[0], z_cp2=zcp[2],
                        z_cp1=zcp[1], z_cp0=zcp[0], z_rd=z_rd,
                        rho=float(rho), d0=float(d0)*scale, ds=float(ds)*scale)

    mass = None
    if wing_mass_g is not None:
        m = float(wing_mass_g) * 1e-3
        mr = integ(lambda r, ints, c: r*c)
        my = integ(lambda r, ints, c: sum(0.5*(y1*y1-y0*y0) for y0,y1 in ints))
        rr = integ(lambda r, ints, c: r*r*c)
        yy = integ(lambda r, ints, c: sum((y1**3-y0**3)/3 for y0,y1 in ints))
        ry = integ(lambda r, ints, c: r*sum(0.5*(y1*y1-y0*y0) for y0,y1 in ints))
        rc, yc = mr/area, my/area
        # Thin uniform lamina; local axes are span r, chord y, surface-normal n.
        I_rr_h, I_yy_h, I_ry_h = m*yy/area, m*rr/area, -m*ry/area
        I_rr_c = I_rr_h - m*yc**2
        I_yy_c = I_yy_h - m*rc**2
        I_ry_c = I_ry_h + m*rc*yc
        mass = dict(mass_kg=m, centroid_r_m=rc, centroid_chord_m=yc,
                    I_rr_hinge=I_rr_h, I_yy_hinge=I_yy_h, I_ry_hinge=I_ry_h,
                    I_rr_com=I_rr_c, I_yy_com=I_yy_c, I_nn_com=I_rr_c+I_yy_c,
                    I_ry_com=I_ry_c)

    return dict(params=params, area_m2=area, aspect_ratio=R/cbar,
                local_polygon_m=local, offset_terms=offset_terms,
                aero_py_exact=compatible, mass_properties=mass)


In [15]:
result = calculate_wing_params(
    polygon, hinge_origin, span_direction, chord_direction,
    length_unit=length_unit, wing_mass_g=wing_mass, rho=air_density,
    d0=shoulder_half_width, ds=stroke_plane_offset,
    quadrature_order=quadrature_order, axis_tolerance=axis_tolerance,
)
p = result['params']
print(f"Area = {result['area_m2']*1e6:.6f} mm^2")
print(f"R = {p.R*1e3:.6f} mm, mean chord = {p.c*1e3:.6f} mm, AR = {result['aspect_ratio']:.6f}")
print(f"Exact match to aero.py pressure-moment form: {result['aero_py_exact']}")
if not result['aero_py_exact']:
    print('WARNING: hinge is not on the leading edge. offset_terms A0..A2 =', result['offset_terms'])
print('\nPython WingParams representation:')
print('WingParams(')
for key, value in asdict(p).items():
    print(f'    {key}={value:.15g},')
print(')')
tutorial_dir = Path.cwd() / 'tutorial' if (Path.cwd() / 'tutorial').is_dir() else Path.cwd()
wing_parameters_path = tutorial_dir / 'wing_parameters.json'
wing_parameters_path.write_text(
    json.dumps(asdict(p), indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
print(f'\nSaved WingParams to: {wing_parameters_path.resolve()}')
if result['mass_properties']:
    print('\nUniform thin-lamina mass properties (SI):')
    for key, value in result['mass_properties'].items(): print(f'  {key}: {value:.12g}')

Area = 1400.000000 mm^2
R = 70.000000 mm, mean chord = 20.000000 mm, AR = 3.500000
Exact match to aero.py pressure-moment form: True

Python WingParams representation:
WingParams(
    R=0.07,
    c=0.02,
    r33=0.25,
    r22=0.333333333333333,
    r11=0.5,
    r00=1,
    z_cp2=0.333333333333333,
    z_cp1=0.5,
    z_cp0=0.999999999999999,
    z_rd=0.999999999999999,
    rho=1.225,
    d0=0.01395,
    ds=0.0189913,
)

Saved WingParams to: /home/robot/rl/FlapSim/tutorial/wing_parameters.json

Uniform thin-lamina mass properties (SI):
  mass_kg: 6.26e-05
  centroid_r_m: 0.035
  centroid_chord_m: 0.01
  I_rr_hinge: 8.34666666667e-09
  I_yy_hinge: 1.02246666667e-07
  I_ry_hinge: -2.191e-08
  I_rr_com: 2.08666666667e-09
  I_yy_com: 2.55616666667e-08
  I_nn_com: 2.76483333333e-08
  I_ry_com: -3.30872245021e-24
